In [ ]:
//@version=6
indicator("Manny's Liquidity Strategy V9", overlay=true, max_lines_count=500, max_labels_count=500, max_boxes_count=500)

// ═══════════════════════════════════════════
// SETTINGS — Easy to understand and adjust
// ═══════════════════════════════════════════

// Levels
showMonthly  = input.bool(true,  "📅 Show Monthly High/Low Lines",        group="📊 Chart Levels")
showWeekly   = input.bool(true,  "📅 Show Weekly High/Low Lines",         group="📊 Chart Levels")
showDaily    = input.bool(true,  "📅 Show Daily High/Low Lines (3 Days)", group="📊 Chart Levels")

// Bias
showWeekBias = input.bool(true,  "📈 Show Weekly Bias (Mon 4AM/8AM GMT)", group="🧭 Bias Levels")
showDayBias  = input.bool(true,  "📈 Show Daily Bias (4AM GMT Candle)",   group="🧭 Bias Levels")

// Features
showEMA      = input.bool(true,  "📉 Show EMA 200 Trend Line",            group="🔧 Features")
showSweeps   = input.bool(true,  "💧 Show Liquidity Sweeps",              group="🔧 Features")
showFVG      = input.bool(true,  "📦 Show Fair Value Gaps (FVG)",         group="🔧 Features")
showDiv      = input.bool(true,  "📐 Show RSI Divergence",                group="🔧 Features")
showBOS      = input.bool(true,  "🔨 Show Break of Structure (BOS)",      group="🔧 Features")
showOB       = input.bool(true,  "🟦 Show Order Blocks",                  group="🔧 Features")
showSignals  = input.bool(true,  "🚦 Show Buy/Sell Signals",              group="🔧 Features")

// Signal Quality
minScore     = input.int(8,      "Minimum Score to Show Signal (out of 12)", group="🎯 Signal Quality", minval=1, maxval=12, tooltip="8 = Good Trade, 9 = Strong Trade, 10+ = A+ Trade. We recommend keeping this at 8 or above.")

// Risk Management
accountSize  = input.float(1000, "💰 Account Size ($)",                   group="💵 Risk Management", tooltip="Enter your total account balance")
riskPercent  = input.float(1.0,  "⚠️ Risk Per Trade (%)",                group="💵 Risk Management", tooltip="1% means you risk 1% of your account per trade. Recommended: 1-2%")
atrLength    = input.int(14,     "ATR Period (Default 14)",               group="💵 Risk Management", tooltip="Used to calculate stop loss and take profit distances. Leave at 14 unless you know what you're doing.")

// ═══════════════════════════════════════════
// TIME HELPERS (GMT)
// ═══════════════════════════════════════════
currentHour = hour(time, "GMT+0")
currentMin  = minute(time, "GMT+0")
currentDOW  = dayofweek(time, "GMT+0")
isMonday    = currentDOW == 2
is4AMCandle = currentHour == 4 and currentMin == 0
is8AMCandle = currentHour == 8 and currentMin == 0
isNewDay    = ta.change(time("D")) != 0
isNewWeek   = ta.change(time("W")) != 0
isNewMonth  = ta.change(time("M")) != 0

// ═══════════════════════════════════════════
// ATR + EMA 200
// ═══════════════════════════════════════════
atr         = ta.atr(atrLength)
ema200      = ta.ema(close, 200)
bullishBias = close > ema200
bearishBias = close < ema200
plot(showEMA ? ema200 : na, "EMA 200 — Trend Direction", color=bullishBias ? color.new(color.green, 0) : color.new(color.red, 0), linewidth=2)

// ═══════════════════════════════════════════
// MONTHLY LEVELS — Purple
// ═══════════════════════════════════════════
month1High = ta.valuewhen(isNewMonth, high[1], 0)
month1Low  = ta.valuewhen(isNewMonth, low[1],  0)
var line mHL = na
var line mLL = na
if isNewMonth and showMonthly
    line.delete(mHL)
    line.delete(mLL)
    mHL := line.new(bar_index, month1High, bar_index + 100, month1High, color=color.new(color.purple, 0), width=2, style=line.style_solid)
    mLL := line.new(bar_index, month1Low,  bar_index + 100, month1Low,  color=color.new(color.purple, 0), width=2, style=line.style_solid)
    label.new(bar_index, month1High, "📅 Monthly High", color=color.new(color.purple, 70), textcolor=color.purple, style=label.style_label_left, size=size.tiny)
    label.new(bar_index, month1Low,  "📅 Monthly Low",  color=color.new(color.purple, 70), textcolor=color.purple, style=label.style_label_left, size=size.tiny)

// Monthly bias — is price heading toward monthly high or low
monthBullAlign = close > month1Low  and bullishBias
monthBearAlign = close < month1High and bearishBias

// ═══════════════════════════════════════════
// WEEKLY LEVELS — Blue
// ═══════════════════════════════════════════
week1High = ta.valuewhen(isNewWeek, high[1], 0)
week1Low  = ta.valuewhen(isNewWeek, low[1],  0)
var line wHL = na
var line wLL = na
if isNewWeek and showWeekly
    line.delete(wHL)
    line.delete(wLL)
    wHL := line.new(bar_index, week1High, bar_index + 100, week1High, color=color.new(color.blue, 0), width=2, style=line.style_solid)
    wLL := line.new(bar_index, week1Low,  bar_index + 100, week1Low,  color=color.new(color.blue, 0), width=2, style=line.style_solid)
    label.new(bar_index, week1High, "📅 Weekly High", color=color.new(color.blue, 70), textcolor=color.blue, style=label.style_label_left, size=size.tiny)
    label.new(bar_index, week1Low,  "📅 Weekly Low",  color=color.new(color.blue, 70), textcolor=color.blue, style=label.style_label_left, size=size.tiny)

// Weekly bias
weekBullish = close > week1Low  and bullishBias
weekBearish = close < week1High and bearishBias

// ═══════════════════════════════════════════
// DAILY LEVELS — Black (3 days fading)
// ═══════════════════════════════════════════
day1High = ta.valuewhen(isNewDay, high[1], 0)
day1Low  = ta.valuewhen(isNewDay, low[1],  0)
day2High = ta.valuewhen(isNewDay, high[1], 1)
day2Low  = ta.valuewhen(isNewDay, low[1],  1)
day3High = ta.valuewhen(isNewDay, high[1], 2)
day3Low  = ta.valuewhen(isNewDay, low[1],  2)
var line d1HL = na
var line d1LL = na
var line d2HL = na
var line d2LL = na
var line d3HL = na
var line d3LL = na
if isNewDay and showDaily
    line.delete(d1HL)
    line.delete(d1LL)
    line.delete(d2HL)
    line.delete(d2LL)
    line.delete(d3HL)
    line.delete(d3LL)
    d1HL := line.new(bar_index, day1High, bar_index + 100, day1High, color=color.new(color.black, 0),  width=2, style=line.style_solid)
    d1LL := line.new(bar_index, day1Low,  bar_index + 100, day1Low,  color=color.new(color.black, 0),  width=2, style=line.style_solid)
    d2HL := line.new(bar_index, day2High, bar_index + 100, day2High, color=color.new(color.black, 40), width=1, style=line.style_dashed)
    d2LL := line.new(bar_index, day2Low,  bar_index + 100, day2Low,  color=color.new(color.black, 40), width=1, style=line.style_dashed)
    d3HL := line.new(bar_index, day3High, bar_index + 100, day3High, color=color.new(color.black, 70), width=1, style=line.style_dotted)
    d3LL := line.new(bar_index, day3Low,  bar_index + 100, day3Low,  color=color.new(color.black, 70), width=1, style=line.style_dotted)
    label.new(bar_index, day1High, "D1H", color=color.new(color.gray, 80), textcolor=color.black, style=label.style_label_left, size=size.tiny)
    label.new(bar_index, day1Low,  "D1L", color=color.new(color.gray, 80), textcolor=color.black, style=label.style_label_left, size=size.tiny)
    label.new(bar_index, day2High, "D2H", color=color.new(color.gray, 80), textcolor=color.black, style=label.style_label_left, size=size.tiny)
    label.new(bar_index, day2Low,  "D2L", color=color.new(color.gray, 80), textcolor=color.black, style=label.style_label_left, size=size.tiny)
    label.new(bar_index, day3High, "D3H", color=color.new(color.gray, 80), textcolor=color.black, style=label.style_label_left, size=size.tiny)
    label.new(bar_index, day3Low,  "D3L", color=color.new(color.gray, 80), textcolor=color.black, style=label.style_label_left, size=size.tiny)

// Daily bias
dayBullish = close > day1Low  and bullishBias
dayBearish = close < day1High and bearishBias

// ═══════════════════════════════════════════
// WEEKLY BIAS — Monday 4AM + 8AM (Gold)
// ═══════════════════════════════════════════
var float weekBiasHigh = na
var float weekBiasLow  = na
var line  wbHL         = na
var line  wbLL         = na

if isMonday and is4AMCandle and showWeekBias
    weekBiasHigh := high
    weekBiasLow  := low

if isMonday and is8AMCandle and showWeekBias
    weekBiasHigh := math.max(weekBiasHigh, high)
    weekBiasLow  := math.min(weekBiasLow,  low)
    line.delete(wbHL)
    line.delete(wbLL)
    wbHL := line.new(bar_index, weekBiasHigh, bar_index + 100, weekBiasHigh, color=color.new(color.yellow, 0), width=2, style=line.style_solid)
    wbLL := line.new(bar_index, weekBiasLow,  bar_index + 100, weekBiasLow,  color=color.new(color.yellow, 0), width=2, style=line.style_solid)
    label.new(bar_index, weekBiasHigh, "⚡ Weekly Bias High", color=color.new(color.yellow, 70), textcolor=color.yellow, style=label.style_label_left, size=size.tiny)
    label.new(bar_index, weekBiasLow,  "⚡ Weekly Bias Low",  color=color.new(color.yellow, 70), textcolor=color.yellow, style=label.style_label_left, size=size.tiny)

// ═══════════════════════════════════════════
// DAILY BIAS — 4AM Candle (Green)
// ═══════════════════════════════════════════
var float dayBiasHigh = na
var float dayBiasLow  = na
var line  dbHL        = na
var line  dbLL        = na

if is4AMCandle and showDayBias
    dayBiasHigh := high
    dayBiasLow  := low
    line.delete(dbHL)
    line.delete(dbLL)
    dbHL := line.new(bar_index, dayBiasHigh, bar_index + 100, dayBiasHigh, color=color.new(color.green, 0), width=1, style=line.style_dashed)
    dbLL := line.new(bar_index, dayBiasLow,  bar_index + 100, dayBiasLow,  color=color.new(color.green, 0), width=1, style=line.style_dashed)
    label.new(bar_index, dayBiasHigh, "🕓 4AM High", color=color.new(color.green, 70), textcolor=color.green, style=label.style_label_left, size=size.tiny)
    label.new(bar_index, dayBiasLow,  "🕓 4AM Low",  color=color.new(color.green, 70), textcolor=color.green, style=label.style_label_left, size=size.tiny)

// ═══════════════════════════════════════════
// LIQUIDITY SWEEPS
// Price wicks beyond previous day H/L but closes back
// ═══════════════════════════════════════════
bullSweepWick = day1Low - low
bearSweepWick = high - day1High
bullishSweep  = low < day1Low   and close > day1Low  and bullSweepWick >= atr * 0.3
bearishSweep  = high > day1High and close < day1High and bearSweepWick >= atr * 0.3

var float lastBullSweepLow  = na
var float lastBearSweepHigh = na
if bullishSweep
    lastBullSweepLow  := low
if bearishSweep
    lastBearSweepHigh := high

plotshape(showSweeps and bullishSweep, "💧 Bull Sweep", shape.triangleup,   location.belowbar, color.new(color.lime, 0), size=size.small, text="Sweep")
plotshape(showSweeps and bearishSweep, "💧 Bear Sweep", shape.triangledown, location.abovebar, color.new(color.red,  0), size=size.small, text="Sweep")

// ═══════════════════════════════════════════
// BREAK OF STRUCTURE (BOS)
// High quality — confirmed close + strong candle
// ═══════════════════════════════════════════
swingLen    = 5
swingHigh   = ta.highest(high, swingLen * 2 + 1)[swingLen]
swingLow    = ta.lowest(low,   swingLen * 2 + 1)[swingLen]

// Strong candle body filter
strongBOSCandle = math.abs(close - open) > atr * 0.7

// BOS confirmed = closes beyond swing + strong candle + bias aligned
bullBOS = close > swingHigh and strongBOSCandle and bullishBias and close[1] <= swingHigh
bearBOS = close < swingLow  and strongBOSCandle and bearishBias and close[1] >= swingLow

// Track last BOS for score
var bool recentBullBOS = false
var bool recentBearBOS = false
var int  bosTimer      = 0

if bullBOS
    recentBullBOS := true
    recentBearBOS := false
    bosTimer      := bar_index

if bearBOS
    recentBearBOS := true
    recentBullBOS := false
    bosTimer      := bar_index

// Reset BOS after 10 bars
if bar_index - bosTimer > 10
    recentBullBOS := false
    recentBearBOS := false

if showBOS and bullBOS
    label.new(bar_index, low  - atr * 0.5, "🔨 BOS ↑", color=color.new(color.green, 20), textcolor=color.white, style=label.style_label_up,   size=size.small)
    line.new(bar_index - 1, swingHigh, bar_index + 10, swingHigh, color=color.new(color.green, 40), width=1, style=line.style_dotted)

if showBOS and bearBOS
    label.new(bar_index, high + atr * 0.5, "🔨 BOS ↓", color=color.new(color.red,   20), textcolor=color.white, style=label.style_label_down, size=size.small)
    line.new(bar_index - 1, swingLow,  bar_index + 10, swingLow,  color=color.new(color.red,   40), width=1, style=line.style_dotted)

// ═══════════════════════════════════════════
// ORDER BLOCKS (OB)
// Last opposing candle before strong impulsive move
// ═══════════════════════════════════════════
impulseLen  = 3
bullImpulse = close > close[1] and close[1] > close[2] and close[2] > close[3]
bearImpulse = close < close[1] and close[1] < close[2] and close[2] < close[3]

// Bullish OB = last bearish candle before bullish impulse
// Bearish OB = last bullish candle before bearish impulse
bullOBCandle = bullImpulse and close[impulseLen] < open[impulseLen] and math.abs(close[impulseLen] - open[impulseLen]) > atr * 0.5
bearOBCandle = bearImpulse and close[impulseLen] > open[impulseLen] and math.abs(close[impulseLen] - open[impulseLen]) > atr * 0.5

var float bullOBTop   = na
var float bullOBBot   = na
var float bearOBTop   = na
var float bearOBBot   = na
var bool  bullOBAlive = false
var bool  bearOBAlive = false
var box   bullOBBox   = na
var box   bearOBBox   = na
var bool  bullOBRespected = false
var bool  bearOBRespected = false
var int   bullOBTouches   = 0
var int   bearOBTouches   = 0

// Create new OB zones
if bullOBCandle and bullishBias and showOB
    bullOBTop       := high[impulseLen]
    bullOBBot       := low[impulseLen]
    bullOBAlive     := true
    bullOBRespected := false
    bullOBTouches   := 0
    box.delete(bullOBBox)
    bullOBBox := box.new(bar_index - impulseLen, bullOBTop, bar_index + 100, bullOBBot, bgcolor=color.new(color.blue, 80), border_color=color.new(color.blue, 30), border_width=1)
    label.new(bar_index - impulseLen, bullOBTop, "🟦 Bull OB", color=color.new(color.blue, 60), textcolor=color.white, style=label.style_label_down, size=size.tiny)

if bearOBCandle and bearishBias and showOB
    bearOBTop       := high[impulseLen]
    bearOBBot       := low[impulseLen]
    bearOBAlive     := true
    bearOBRespected := false
    bearOBTouches   := 0
    box.delete(bearOBBox)
    bearOBBox := box.new(bar_index - impulseLen, bearOBTop, bar_index + 100, bearOBBot, bgcolor=color.new(color.orange, 80), border_color=color.new(color.orange, 30), border_width=1)
    label.new(bar_index - impulseLen, bearOBBot, "🟧 Bear OB", color=color.new(color.orange, 60), textcolor=color.white, style=label.style_label_up, size=size.tiny)

// Check if price returns to OB zone — first touch only
if bullOBAlive and bullOBTouches == 0
    if low <= bullOBTop and close >= bullOBBot
        bullOBRespected := true
        bullOBTouches   := bullOBTouches + 1
        box.set_bgcolor(bullOBBox, color.new(color.green, 70))
        box.set_border_color(bullOBBox, color.new(color.green, 20))
        label.new(bar_index, bullOBBot, "✅ OB Respected", color=color.new(color.green, 30), textcolor=color.white, style=label.style_label_up, size=size.tiny)

if bearOBAlive and bearOBTouches == 0
    if high >= bearOBBot and close <= bearOBTop
        bearOBRespected := true
        bearOBTouches   := bearOBTouches + 1
        box.set_bgcolor(bearOBBox, color.new(color.red, 70))
        box.set_border_color(bearOBBox, color.new(color.red, 20))
        label.new(bar_index, bearOBTop, "✅ OB Respected", color=color.new(color.red, 30), textcolor=color.white, style=label.style_label_down, size=size.tiny)

// Invalidate OB if price closes through it
if bullOBAlive and close < bullOBBot
    bullOBAlive     := false
    bullOBRespected := false
    box.set_bgcolor(bullOBBox, color.new(color.gray, 90))

if bearOBAlive and close > bearOBTop
    bearOBAlive     := false
    bearOBRespected := false
    box.set_bgcolor(bearOBBox, color.new(color.gray, 90))

// ═══════════════════════════════════════════
// HIGH QUALITY FVG
// Strong candle + bias aligned + stays visible
// ═══════════════════════════════════════════
candleBody   = math.abs(close[1] - open[1])
strongCandle = candleBody > atr * 1.0

bullFVGraw = low > high[2] and strongCandle and close[1] > open[1] and bullishBias
bearFVGraw = high < low[2] and strongCandle and close[1] < open[1] and bearishBias

var float bullFVGTop      = na
var float bullFVGBot      = na
var float bearFVGTop      = na
var float bearFVGBot      = na
var bool  bullFVGAlive    = false
var bool  bearFVGAlive    = false
var box   bullFVGBox      = na
var box   bearFVGBox      = na
var bool  bullFVGRespected = false
var bool  bearFVGRespected = false

if bullFVGraw and showFVG
    bullFVGTop      := low
    bullFVGBot      := high[2]
    bullFVGAlive    := true
    bullFVGRespected := false
    box.delete(bullFVGBox)
    bullFVGBox := box.new(bar_index - 2, bullFVGTop, bar_index + 100, bullFVGBot, bgcolor=color.new(color.green, 85), border_color=color.new(color.green, 40), border_width=1)
    label.new(bar_index, bullFVGBot, "📦 FVG", color=color.new(color.green, 50), textcolor=color.white, style=label.style_label_up, size=size.tiny)

if bearFVGraw and showFVG
    bearFVGTop      := low[2]
    bearFVGBot      := high
    bearFVGAlive    := true
    bearFVGRespected := false
    box.delete(bearFVGBox)
    bearFVGBox := box.new(bar_index - 2, bearFVGTop, bar_index + 100, bearFVGBot, bgcolor=color.new(color.red, 85), border_color=color.new(color.red, 40), border_width=1)
    label.new(bar_index, bearFVGTop, "📦 FVG", color=color.new(color.red, 50), textcolor=color.white, style=label.style_label_down, size=size.tiny)

// FVG respected = price touched AND closed back in direction
if bullFVGAlive and low <= bullFVGTop and close >= bullFVGTop
    bullFVGRespected := true
    box.set_bgcolor(bullFVGBox, color.new(color.green, 70))
    box.set_border_color(bullFVGBox, color.new(color.green, 20))
    label.new(bar_index, bullFVGBot, "✅ FVG Respected", color=color.new(color.green, 30), textcolor=color.white, style=label.style_label_up, size=size.tiny)

if bearFVGAlive and high >= bearFVGBot and close <= bearFVGBot
    bearFVGRespected := true
    box.set_bgcolor(bearFVGBox, color.new(color.red, 70))
    box.set_border_color(bearFVGBox, color.new(color.red, 20))
    label.new(bar_index, bearFVGTop, "✅ FVG Respected", color=color.new(color.red, 30), textcolor=color.white, style=label.style_label_down, size=size.tiny)

// Invalidate FVG if price closes through it
if bullFVGAlive and close < bullFVGBot
    bullFVGAlive    := false
    bullFVGRespected := false
    box.set_bgcolor(bullFVGBox, color.new(color.gray, 90))

if bearFVGAlive and close > bearFVGTop
    bearFVGAlive    := false
    bearFVGRespected := false
    box.set_bgcolor(bearFVGBox, color.new(color.gray, 90))

// ═══════════════════════════════════════════
// RSI DIVERGENCE — Strong + Hidden only
// ═══════════════════════════════════════════
rsi = ta.rsi(close, 14)
lkb = 7

priceLL = ta.lowest(low,   lkb) < ta.lowest(low[lkb],   lkb)
priceHL = ta.lowest(low,   lkb) > ta.lowest(low[lkb],   lkb)
priceHH = ta.highest(high, lkb) > ta.highest(high[lkb], lkb)
priceLH = ta.highest(high, lkb) < ta.highest(high[lkb], lkb)

rsiHL = ta.lowest(rsi,  lkb) > ta.lowest(rsi[lkb],  lkb)
rsiLL = ta.lowest(rsi,  lkb) < ta.lowest(rsi[lkb],  lkb)
rsiLH = ta.highest(rsi, lkb) < ta.highest(rsi[lkb], lkb)
rsiHH = ta.highest(rsi, lkb) > ta.highest(rsi[lkb], lkb)

rsiBullExtreme = rsi < 40
rsiBearExtreme = rsi > 60

strongBullDiv = priceLL and rsiHL
hiddenBullDiv = priceHL and rsiLL
strongBearDiv = priceHH and rsiLH
hiddenBearDiv = priceLH and rsiHH

anyBullDiv = strongBullDiv or hiddenBullDiv
anyBearDiv = strongBearDiv or hiddenBearDiv

if showDiv and strongBullDiv
    label.new(bar_index, low  - atr * 0.3, "◆ Strong Bull Div", color=color.new(color.green,  0), textcolor=color.white, style=label.style_label_up,   size=size.tiny)
if showDiv and hiddenBullDiv
    label.new(bar_index, low  - atr * 0.3, "◇ Hidden Bull Div", color=color.new(color.aqua,   0), textcolor=color.white, style=label.style_label_up,   size=size.tiny)
if showDiv and strongBearDiv
    label.new(bar_index, high + atr * 0.3, "◆ Strong Bear Div", color=color.new(color.red,    0), textcolor=color.white, style=label.style_label_down, size=size.tiny)
if showDiv and hiddenBearDiv
    label.new(bar_index, high + atr * 0.3, "◇ Hidden Bear Div", color=color.new(color.orange, 0), textcolor=color.white, style=label.style_label_down, size=size.tiny)

// ═══════════════════════════════════════════
// MACD
// ═══════════════════════════════════════════
[macdLine, signalLine, histLine] = ta.macd(close, 12, 26, 9)
macdBullish  = macdLine > signalLine
macdBearish  = macdLine < signalLine
macdMomentum = histLine > histLine[1]

// ═══════════════════════════════════════════
// CONFLUENCE SCORE — 12 POINTS
// ═══════════════════════════════════════════

// LONG SCORE
longScore = 0
longScore := longScore + (bullishBias                            ? 1 : 0)  // 1.  EMA trend
longScore := longScore + (monthBullAlign                         ? 1 : 0)  // 2.  Monthly bias
longScore := longScore + (weekBullish                            ? 1 : 0)  // 3.  Weekly bias
longScore := longScore + (dayBullish                             ? 1 : 0)  // 4.  Daily bias
longScore := longScore + (bullishSweep                           ? 1 : 0)  // 5.  Liquidity sweep
longScore := longScore + (recentBullBOS                          ? 1 : 0)  // 6.  Break of structure
longScore := longScore + (bullOBRespected                        ? 1 : 0)  // 7.  Order block respected
longScore := longScore + (bullFVGRespected                       ? 1 : 0)  // 8.  FVG respected
longScore := longScore + (strongBullDiv ? 2 : hiddenBullDiv ? 1 : 0)      // 9-10. Divergence
longScore := longScore + (macdBullish                            ? 1 : 0)  // 11. MACD
longScore := longScore + (rsiBullExtreme                         ? 1 : 0)  // 12. RSI extreme

// SHORT SCORE
shortScore = 0
shortScore := shortScore + (bearishBias                          ? 1 : 0)
shortScore := shortScore + (monthBearAlign                       ? 1 : 0)
shortScore := shortScore + (weekBearish                          ? 1 : 0)
shortScore := shortScore + (dayBearish                           ? 1 : 0)
shortScore := shortScore + (bearishSweep                         ? 1 : 0)
shortScore := shortScore + (recentBearBOS                        ? 1 : 0)
shortScore := shortScore + (bearOBRespected                      ? 1 : 0)
shortScore := shortScore + (bearFVGRespected                     ? 1 : 0)
shortScore := shortScore + (strongBearDiv ? 2 : hiddenBearDiv ? 1 : 0)
shortScore := shortScore + (macdBearish                          ? 1 : 0)
shortScore := shortScore + (rsiBearExtreme                       ? 1 : 0)

// ═══════════════════════════════════════════
// CONFIDENCE LABEL
// ═══════════════════════════════════════════
confidenceLabel(score) =>
    score >= 11 ? "🔥 A+ PERFECT"  :
     score >= 9  ? "💪 STRONG"      :
     score >= 8  ? "✅ GOOD TRADE"  :
                   "⏳ WAIT"

confidenceColor(score) =>
    score >= 11 ? color.new(color.yellow, 0) :
     score >= 9  ? color.new(color.green,  0) :
     score >= 8  ? color.new(color.teal,   0) :
                   color.new(color.gray,   0)

// ═══════════════════════════════════════════
// RISK MANAGEMENT — 1:3 only
// ═══════════════════════════════════════════
riskAmount = accountSize * (riskPercent / 100)
slBuffer   = longScore >= 9 or shortScore >= 9 ? atr * 0.2 : atr * 0.5

longSL    = not na(lastBullSweepLow)  ? lastBullSweepLow  - slBuffer : close - atr * 1.5
longRisk  = math.max(close - longSL,  0.0001)
longTP    = close + longRisk * 3
longSize  = riskAmount / longRisk

shortSL   = not na(lastBearSweepHigh) ? lastBearSweepHigh + slBuffer : close + atr * 1.5
shortRisk = math.max(shortSL - close, 0.0001)
shortTP   = close - shortRisk * 3
shortSize = riskAmount / shortRisk

// ═══════════════════════════════════════════
// ENTRY SIGNAL LABELS — full breakdown
// ═══════════════════════════════════════════
bullishEntry = longScore  >= minScore
bearishEntry = shortScore >= minScore

longConditions =
     "⬆ BUY SIGNAL — " + str.tostring(longScore) + "/12\n" +
     confidenceLabel(longScore) + "\n" +
     "━━━━━━━━━━━━━━━━━━━\n" +
     "CORE CONDITIONS:\n" +
     (bullishBias      ? "✅ EMA 200: Bullish Trend\n"        : "❌ EMA 200: Not Bullish\n")       +
     (weekBullish      ? "✅ Weekly Bias: Confirmed\n"         : "❌ Weekly Bias: Not Confirmed\n") +
     (dayBullish       ? "✅ Daily Bias: Confirmed\n"          : "❌ Daily Bias: Not Confirmed\n")  +
     (recentBullBOS    ? "✅ Break of Structure: Yes\n"        : "❌ Break of Structure: No\n")     +
     (bullishSweep     ? "✅ Liquidity Sweep: Yes\n"           : "❌ Liquidity Sweep: No\n")        +
     (anyBullDiv       ? "✅ RSI Divergence: " + (strongBullDiv ? "Strong\n" : "Hidden\n") : "❌ RSI Divergence: None\n") +
     "━━━━━━━━━━━━━━━━━━━\n" +
     "CONFLUENCE:\n" +
     (monthBullAlign   ? "✅ Monthly Bias: Aligned\n"          : "➖ Monthly Bias: Not Set\n")      +
     (bullOBRespected  ? "✅ Order Block: Respected\n"         : "➖ Order Block: Not Hit\n")       +
     (bullFVGRespected ? "✅ FVG: Respected\n"                 : "➖ FVG: Not Respected\n")         +
     (macdBullish      ? "✅ MACD: Bullish\n"                  : "➖ MACD: Not Bullish\n")          +
     (rsiBullExtreme   ? "✅ RSI Oversold: " + str.tostring(math.round(rsi, 1)) + "\n" : "➖ RSI: " + str.tostring(math.round(rsi, 1)) + "\n") +
     "━━━━━━━━━━━━━━━━━━━\n" +
     "TRADE LEVELS:\n" +
     "🔴 Stop Loss:  $" + str.tostring(math.round(longSL,         2)) + "\n" +
     "🟢 Take Profit: $" + str.tostring(math.round(longTP,        2)) + "\n" +
     "💰 Risk:       $" + str.tostring(math.round(riskAmount,     2)) + "\n" +
     "🎯 Target:     $" + str.tostring(math.round(riskAmount * 3, 2)) + "\n" +
     "📊 Size:        " + str.tostring(math.round(longSize,       4)) + " units"

shortConditions =
     "⬇ SELL SIGNAL — " + str.tostring(shortScore) + "/12\n" +
     confidenceLabel(shortScore) + "\n" +
     "━━━━━━━━━━━━━━━━━━━\n" +
     "CORE CONDITIONS:\n" +
     (bearishBias      ? "✅ EMA 200: Bearish Trend\n"         : "❌ EMA 200: Not Bearish\n")      +
     (weekBearish      ? "✅ Weekly Bias: Confirmed\n"          : "❌ Weekly Bias: Not Confirmed\n") +
     (dayBearish       ? "✅ Daily Bias: Confirmed\n"           : "❌ Daily Bias: Not Confirmed\n") +
     (recentBearBOS    ? "✅ Break of Structure: Yes\n"         : "❌ Break of Structure: No\n")    +
     (bearishSweep     ? "✅ Liquidity Sweep: Yes\n"            : "❌ Liquidity Sweep: No\n")       +
     (anyBearDiv       ? "✅ RSI Divergence: " + (strongBearDiv ? "Strong\n" : "Hidden\n") : "❌ RSI Divergence: None\n") +
     "━━━━━━━━━━━━━━━━━━━\n" +
     "CONFLUENCE:\n" +
     (monthBearAlign   ? "✅ Monthly Bias: Aligned\n"           : "➖ Monthly Bias: Not Set\n")     +
     (bearOBRespected  ? "✅ Order Block: Respected\n"          : "➖ Order Block: Not Hit\n")      +
     (bearFVGRespected ? "✅ FVG: Respected\n"                  : "➖ FVG: Not Respected\n")        +
     (macdBearish      ? "✅ MACD: Bearish\n"                   : "➖ MACD: Not Bearish\n")         +
     (rsiBearExtreme   ? "✅ RSI Overbought: " + str.tostring(math.round(rsi, 1)) + "\n" : "➖ RSI: " + str.tostring(math.round(rsi, 1)) + "\n") +
     "━━━━━━━━━━━━━━━━━━━\n" +
     "TRADE LEVELS:\n" +
     "🔴 Stop Loss:   $" + str.tostring(math.round(shortSL,        2)) + "\n" +
     "🟢 Take Profit: $" + str.tostring(math.round(shortTP,        2)) + "\n" +
     "💰 Risk:        $" + str.tostring(math.round(riskAmount,     2)) + "\n" +
     "🎯 Target:      $" + str.tostring(math.round(riskAmount * 3, 2)) + "\n" +
     "📊 Size:         " + str.tostring(math.round(shortSize,      4)) + " units"

if showSignals and bullishEntry
    label.new(bar_index, low, longConditions, color=color.new(color.green, 10), textcolor=color.white, style=label.style_label_up, size=size.small)
    line.new(bar_index, longSL,  bar_index + 40, longSL,  color=color.red,   width=2, style=line.style_dashed)
    line.new(bar_index, longTP,  bar_index + 40, longTP,  color=color.green, width=2, style=line.style_dashed)
    label.new(bar_index + 40, longSL,  "🔴 SL", color=color.new(color.red,   0), textcolor=color.white, style=label.style_label_left, size=size.tiny)
    label.new(bar_index + 40, longTP,  "🟢 TP", color=color.new(color.green, 0), textcolor=color.white, style=label.style_label_left, size=size.tiny)

if showSignals and bearishEntry
    label.new(bar_index, high, shortConditions, color=color.new(color.red, 10), textcolor=color.white, style=label.style_label_down, size=size.small)
    line.new(bar_index, shortSL, bar_index + 40, shortSL, color=color.red,   width=2, style=line.style_dashed)
    line.new(bar_index, shortTP, bar_index + 40, shortTP, color=color.green, width=2, style=line.style_dashed)
    label.new(bar_index + 40, shortSL, "🔴 SL", color=color.new(color.red,   0), textcolor=color.white, style=label.style_label_left, size=size.tiny)
    label.new(bar_index + 40, shortTP, "🟢 TP", color=color.new(color.green, 0), textcolor=color.white, style=label.style_label_left, size=size.tiny)

// ═══════════════════════════════════════════
// DASHBOARD — Clean and user friendly
// ═══════════════════════════════════════════
var table dash = table.new(position.top_right, 2, 16, bgcolor=color.new(color.black, 60), border_color=color.new(color.gray, 50), border_width=1)

// Header
table.cell(dash, 0, 0,  "📊 CONDITION",      text_color=color.white, text_size=size.small, bgcolor=color.new(color.navy,  20))
table.cell(dash, 1, 0,  "STATUS",            text_color=color.white, text_size=size.small, bgcolor=color.new(color.navy,  20))

// Core conditions
table.cell(dash, 0, 1,  "EMA 200 Trend",     text_color=color.white, text_size=size.small)
table.cell(dash, 1, 1,  bullishBias ? "🟢 BULL" : "🔴 BEAR",          text_color=bullishBias ? color.green : color.red,    text_size=size.small)
table.cell(dash, 0, 2,  "Monthly Bias",      text_color=color.white, text_size=size.small)
table.cell(dash, 1, 2,  monthBullAlign ? "🟢 BULL" : monthBearAlign ? "🔴 BEAR" : "⏳ WAIT", text_color=monthBullAlign ? color.green : monthBearAlign ? color.red : color.yellow, text_size=size.small)
table.cell(dash, 0, 3,  "Weekly Bias",       text_color=color.white, text_size=size.small)
table.cell(dash, 1, 3,  weekBullish ? "🟢 BULL" : weekBearish ? "🔴 BEAR" : "⏳ WAIT",        text_color=weekBullish ? color.green : weekBearish ? color.red : color.yellow,       text_size=size.small)
table.cell(dash, 0, 4,  "Daily Bias",        text_color=color.white, text_size=size.small)
table.cell(dash, 1, 4,  dayBullish ? "🟢 BULL" : dayBearish ? "🔴 BEAR" : "⏳ WAIT",          text_color=dayBullish ? color.green : dayBearish ? color.red : color.yellow,         text_size=size.small)
table.cell(dash, 0, 5,  "Break of Structure",text_color=color.white, text_size=size.small)
table.cell(dash, 1, 5,  recentBullBOS ? "🟢 BULL BOS" : recentBearBOS ? "🔴 BEAR BOS" : "⏳ NONE", text_color=recentBullBOS ? color.green : recentBearBOS ? color.red : color.gray, text_size=size.small)
table.cell(dash, 0, 6,  "Liquidity Sweep",   text_color=color.white, text_size=size.small)
table.cell(dash, 1, 6,  bullishSweep ? "🟢 BULL" : bearishSweep ? "🔴 BEAR" : "⏳ NONE",      text_color=bullishSweep or bearishSweep ? color.green : color.gray,                  text_size=size.small)
table.cell(dash, 0, 7,  "RSI Divergence",    text_color=color.white, text_size=size.small)
table.cell(dash, 1, 7,  strongBullDiv ? "🟢 Strong Bull" : hiddenBullDiv ? "🟢 Hidden Bull" : strongBearDiv ? "🔴 Strong Bear" : hiddenBearDiv ? "🔴 Hidden Bear" : "⏳ NONE",     text_color=anyBullDiv ? color.green : anyBearDiv ? color.red : color.gray, text_size=size.small)

// Confluence conditions
table.cell(dash, 0, 8,  "── CONFLUENCE ──",  text_color=color.gray,  text_size=size.tiny,  bgcolor=color.new(color.gray, 80))
table.cell(dash, 1, 8,  "──────────────",    text_color=color.gray,  text_size=size.tiny,  bgcolor=color.new(color.gray, 80))
table.cell(dash, 0, 9,  "Order Block",       text_color=color.white, text_size=size.small)
table.cell(dash, 1, 9,  bullOBRespected ? "✅ BULL RESP" : bearOBRespected ? "✅ BEAR RESP" : bullOBAlive ? "🟦 BULL LIVE" : bearOBAlive ? "🟧 BEAR LIVE" : "⏳ NONE",             text_color=bullOBRespected or bearOBRespected ? color.green : bullOBAlive or bearOBAlive ? color.yellow : color.gray, text_size=size.small)
table.cell(dash, 0, 10, "Fair Value Gap",    text_color=color.white, text_size=size.small)
table.cell(dash, 1, 10, bullFVGRespected ? "✅ BULL RESP" : bearFVGRespected ? "✅ BEAR RESP" : bullFVGAlive ? "📦 BULL LIVE" : bearFVGAlive ? "📦 BEAR LIVE" : "⏳ NONE",         text_color=bullFVGRespected or bearFVGRespected ? color.green : bullFVGAlive or bearFVGAlive ? color.yellow : color.gray, text_size=size.small)
table.cell(dash, 0, 11, "MACD",             text_color=color.white, text_size=size.small)
table.cell(dash, 1, 11, macdBullish ? "🟢 BULL" : "🔴 BEAR",          text_color=macdBullish ? color.green : color.red,    text_size=size.small)
table.cell(dash, 0, 12, "RSI Value",        text_color=color.white, text_size=size.small)
table.cell(dash, 1, 12, str.tostring(math.round(rsi, 1)) + (rsiBullExtreme ? " 🟢 Oversold" : rsiBearExtreme ? " 🔴 Overbought" : " ⚪ Neutral"), text_color=rsiBullExtreme ? color.green : rsiBearExtreme ? color.red : color.white, text_size=size.small)

// Risk info
table.cell(dash, 0, 13, "── RISK INFO ──",   text_color=color.gray,  text_size=size.tiny,  bgcolor=color.new(color.gray, 80))
table.cell(dash, 1, 13, "──────────────",    text_color=color.gray,  text_size=size.tiny,  bgcolor=color.new(color.gray, 80))
table.cell(dash, 0, 14, "Risk / Target",     text_color=color.white, text_size=size.small)
table.cell(dash, 1, 14, "💰 $" + str.tostring(math.round(riskAmount, 2)) + " → 🎯 $" + str.tostring(math.round(riskAmount * 3, 2)), text_color=color.yellow, text_size=size.small)

// Signal
table.cell(dash, 0, 15, "🚦 SIGNAL",         text_color=color.white, text_size=size.small, bgcolor=color.new(color.gray, 40))
table.cell(dash, 1, 15,
     bullishEntry ? "⬆ BUY " + str.tostring(longScore)  + "/12 " + confidenceLabel(longScore)  :
     bearishEntry ? "⬇ SELL " + str.tostring(shortScore) + "/12 " + confidenceLabel(shortScore) :
     "⏳ WAIT — " + str.tostring(math.max(longScore, shortScore)) + "/12",
     text_color=bullishEntry ? color.green : bearishEntry ? color.red : color.yellow,
     text_size=size.small)

In [1]:
import pandas as pd

Read a .csv from a URL with pandas

target website : https://football-data.co.uk/data.php

In [3]:
#reading 1 csv file from the website 
df_premier25 = pd.read_csv('https://football-data.co.uk/mmz4281/2526/E0.csv')

In [4]:
#show the df
df_premier25

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA
0,E0,15/08/2025,20:00,Liverpool,Bournemouth,4,2,H,1,0,...,2.03,1.78,2.07,1.85,2.03,1.88,1.94,1.76,2.14,1.86
1,E0,16/08/2025,12:30,Aston Villa,Newcastle,0,0,D,0,0,...,2.05,1.80,2.02,1.89,2.06,1.80,1.95,1.74,2.14,1.86
2,E0,16/08/2025,15:00,Brighton,Fulham,1,1,D,0,0,...,1.83,2.03,1.93,2.00,1.84,2.03,1.80,1.96,1.91,2.08
3,E0,16/08/2025,15:00,Sunderland,West Ham,3,0,H,0,0,...,1.95,1.90,1.97,1.95,1.95,1.94,1.86,1.78,2.02,1.97
4,E0,16/08/2025,15:00,Tottenham,Burnley,3,0,H,1,0,...,1.98,1.88,1.99,1.93,1.98,1.91,1.88,1.83,2.07,1.92
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,E0,14/03/2026,20:00,West Ham,Man City,1,1,D,1,1,...,2.05,1.80,NaN,NaN,2.05,1.83,1.99,1.79,2.15,1.85
296,E0,15/03/2026,14:00,Crystal Palace,Leeds,0,0,D,0,0,...,1.93,1.93,NaN,NaN,1.93,1.93,1.89,1.88,2.01,1.97
297,E0,15/03/2026,14:00,Man United,Aston Villa,3,1,H,0,0,...,1.90,1.95,NaN,NaN,1.93,1.96,1.88,1.90,1.96,2.02
298,E0,15/03/2026,14:00,Nott'm Forest,Fulham,0,0,D,0,0,...,2.00,1.85,NaN,NaN,2.00,1.91,1.92,1.85,2.04,1.93


In [5]:
#rename columns 
df_premier25.rename(columns={'FTHG':'home_goals',
                             'FTAG':'away_goals'}, inplace=True)

In [6]:
#show the df
df_premier25

,Div,Date,Time,HomeTeam,AwayTeam,home_goals,away_goals,FTR,HTHG,HTAG,...,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA
0,E0,15/08/2025,20:00,Liverpool,Bournemouth,4,2,H,1,0,...,2.03,1.78,2.07,1.85,2.03,1.88,1.94,1.76,2.14,1.86
1,E0,16/08/2025,12:30,Aston Villa,Newcastle,0,0,D,0,0,...,2.05,1.80,2.02,1.89,2.06,1.80,1.95,1.74,2.14,1.86
2,E0,16/08/2025,15:00,Brighton,Fulham,1,1,D,0,0,...,1.83,2.03,1.93,2.00,1.84,2.03,1.80,1.96,1.91,2.08
3,E0,16/08/2025,15:00,Sunderland,West Ham,3,0,H,0,0,...,1.95,1.90,1.97,1.95,1.95,1.94,1.86,1.78,2.02,1.97
4,E0,16/08/2025,15:00,Tottenham,Burnley,3,0,H,1,0,...,1.98,1.88,1.99,1.93,1.98,1.91,1.88,1.83,2.07,1.92
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,E0,14/03/2026,20:00,West Ham,Man City,1,1,D,1,1,...,2.05,1.80,NaN,NaN,2.05,1.83,1.99,1.79,2.15,1.85
296,E0,15/03/2026,14:00,Crystal Palace,Leeds,0,0,D,0,0,...,1.93,1.93,NaN,NaN,1.93,1.93,1.89,1.88,2.01,1.97
297,E0,15/03/2026,14:00,Man United,Aston Villa,3,1,H,0,0,...,1.90,1.95,NaN,NaN,1.93,1.96,1.88,1.90,1.96,2.02
298,E0,15/03/2026,14:00,Nott'm Forest,Fulham,0,0,D,0,0,...,2.00,1.85,NaN,NaN,2.00,1.91,1.92,1.85,2.04,1.93


Read .csv file from multiple URLs with Pandas

-https://football-data.co.uk/mmz4281/2526/E0.csv

-https://football-data.co.uk/mmz4281/2526/E1.csv

Link: root + season + league

In [8]:
#link structure
"https://football-data.co.uk/mmz4281/" + "2526" + "/" + "E0" +".csv"


'https://football-data.co.uk/mmz4281/2526/E0.csv'

In [9]:
#create a root variable 
root = "https://football-data.co.uk/mmz4281/"

Multiple Leagues 

-https://football-data.co.uk/mmz4281/2526/E0.csv

-https://football-data.co.uk/mmz4281/2526/E1.csv

-https://football-data.co.uk/mmz4281/2526/E2.csv

-https://football-data.co.uk/mmz4281/2526/E3.csv

-https://football-data.co.uk/mmz4281/2526/EC.csv

In [10]:
#create list of leagues
leagues = ['E0', 'E1', 'E2', 'E3', 'EC']
frames = []#create an empty list 
#looping through leagues, read multiple csv and append it into a list
for league in leagues:
    df = pd.read_csv( root + '2526' + '/' + league + '.csv')
    frames.append(df)#store each df into the frame list adding all legues into one list

In [12]:
len(frames)#this shows use the length of the file such we have 5 files inside this veraible to it works

5

In [17]:
#show 1st, 2nd and 3rf element 
frames[4]

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA
0,EC,09/08/2025,15:00,Altrincham,Aldershot,3,2,H,2,0,...,1.90,1.90,1.92,1.93,1.90,1.90,1.86,1.84,2.00,1.99
1,EC,09/08/2025,15:00,Boreham Wood,Rochdale,0,2,A,0,0,...,2.00,1.80,2.05,1.82,2.02,1.80,1.96,1.75,2.06,1.89
2,EC,09/08/2025,15:00,Brackley Town,Eastleigh,1,0,H,1,0,...,2.00,1.80,2.14,1.72,2.03,1.80,1.96,1.75,2.06,1.84
3,EC,09/08/2025,15:00,Braintree Town,Halifax,3,0,H,2,0,...,1.93,1.88,1.97,1.88,1.93,1.88,1.88,1.82,1.97,1.98
4,EC,09/08/2025,15:00,Gateshead,Southend,0,3,A,0,1,...,1.83,1.98,1.67,2.19,1.83,2.04,1.75,1.96,1.84,2.12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
441,EC,14/03/2026,15:00,Tamworth,Carlisle,2,1,H,0,0,...,1.95,1.85,NaN,NaN,1.95,1.85,1.89,1.81,2.02,1.93
442,EC,14/03/2026,15:00,Truro,Hartlepool,0,1,A,0,0,...,1.85,1.95,NaN,NaN,1.85,1.95,1.81,1.88,1.91,2.03
443,EC,14/03/2026,17:30,Eastleigh,Rochdale,1,3,A,0,1,...,1.93,1.88,NaN,NaN,1.93,1.88,1.89,1.81,2.00,1.91
444,EC,14/03/2026,17:30,Morecambe,Braintree Town,1,1,D,0,0,...,1.83,1.98,NaN,NaN,1.85,1.98,1.79,1.91,1.86,2.14


Multiple Seasons

In [31]:
for season in range(15, 26):# if the season is between 15 - 26
    print(str(season) + str(season+1))#print all season and the following season to come 

1516
1617
1718
1819
1920
2021
2122
2223
2324
2425
2526


In [34]:
#create list of leagues
leagues = ['E0', 'E2', 'E3']
frames = []#create an empty list 
#looping through range of seasonsand insert new columns
for league in leagues:
    for season in range(15, 26):#for loop to go through the files and get the seasons
        df = pd.read_csv( root + str(season) + str(season+1) + '/' + league + '.csv')
        df.insert(1, 'season', season)#so add a new column called season by inserting on the df and then show onlt the season year
        frames.append(df)#store each df into the frame list adding all legues into one list

In [35]:
#total frames (11 seasons X 3 leagues)
len(frames)

33

In [41]:
#show 1st and last element (new season column was created)
frames[32]

,Div,season,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,...,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA,Unnamed: 132
0,E3,25,02/08/2025,15:00,Accrington,Gillingham,1,1,D,0,...,1.88,1.76,2.12,1.98,1.88,1.92,1.83,2.09,1.88,NaN
1,E3,25,02/08/2025,15:00,Barnet,Fleetwood Town,0,2,A,0,...,1.85,2.29,1.65,2.08,1.85,2.00,1.74,2.04,1.93,NaN
2,E3,25,02/08/2025,15:00,Bristol Rvs,Harrogate,0,1,A,0,...,2.03,1.85,2.00,1.85,2.03,1.78,1.96,1.88,2.12,NaN
3,E3,25,02/08/2025,15:00,Cambridge,Cheltenham,1,0,H,0,...,1.88,1.88,1.97,1.98,1.88,1.90,1.83,1.98,2.00,NaN
4,E3,25,02/08/2025,15:00,Chesterfield,Barrow,1,0,H,1,...,2.03,1.79,2.06,1.80,2.07,1.73,2.02,1.87,2.13,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
434,E3,25,14/03/2026,15:00,Crewe,Walsall,0,3,A,0,...,1.85,NaN,NaN,2.00,1.93,1.87,1.85,2.09,1.90,NaN
435,E3,25,14/03/2026,15:00,Fleetwood Town,Tranmere,0,0,D,0,...,1.88,NaN,NaN,1.98,1.88,1.93,1.78,2.07,1.88,NaN
436,E3,25,14/03/2026,15:00,Oldham,Grimsby,1,0,H,0,...,1.98,NaN,NaN,1.88,1.98,1.81,1.93,1.90,2.07,NaN
437,E3,25,14/03/2026,15:00,Shrewsbury,Cheltenham,0,2,A,0,...,1.83,NaN,NaN,2.03,1.83,1.93,1.79,2.14,1.86,NaN
